# Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Configuration](#2-configuration)
3. [Reproducibility and Device Setup](#3-reproducibility-and-device-setup)
4. [Dataset Discovery and Metadata Cache](#4-dataset-discovery-and-metadata-cache)
5. [Dependencies and Imports](#5-dependencies-and-imports)
6. [Data Loading and Preprocessing](#6-data-loading-and-preprocessing)
7. [Model Architecture](#7-model-architecture)
8. [Experiment Tracking](#8-experiment-tracking)
9. [Training Utilities](#9-training-utilities)
10. [Training Loop](#10-training-loop)
11. [Evaluation](#11-evaluation)
12. [Visualization of Predictions](#12-visualization-of-predictions)
13. [Inference Examples](#13-inference-examples)

# Tampered Image Detection and Localization (vK.10.3)

This Kaggle-first notebook presents a complete assignment submission for tampered image
detection and tampered region localization.

**Engineering upgrades in vK.10**
- Centralized CONFIG dictionary for all hyperparameters
- Full reproducibility seeding (Python, NumPy, PyTorch CPU/CUDA)
- Automatic Mixed Precision (AMP) for faster training
- Three-file checkpoint system with automatic resume
- Early stopping based on tampered-only Dice coefficient
- Tampered-only metric reporting to address metric inflation
- GPU diagnostics and VRAM-based batch size auto-adjustment
- Metadata caching to skip redundant dataset scanning
- Optimized DataLoaders (persistent workers, seeded workers, drop_last)

**Notebook deliverables**
- Image-level tamper detection through the classifier head
- Pixel-level tampered region localization through the segmentation branch
- Reproducible Kaggle-first execution with Colab/Drive fallback
- Qualitative visual evidence showing predicted masks and overlays


## Project Objectives: Fulfilled vs Remaining

| Requirement | Status | Evidence |
|---|---|---|
| Dataset: authentic + tampered + masks | Fulfilled | CASIA dataset with IMAGE/MASK dirs |
| Model performs detection + localization | Fulfilled | UNetWithClassifier dual-head |
| Evaluation with Dice / IoU / F1 | Fulfilled | Tampered-only and all-sample metrics |
| Visual results (Original, GT, Pred, Overlay) | Fulfilled | Submission prediction grid |
| Single notebook | Fulfilled | All code in one notebook |
| Reproducibility | Fulfilled | Full seeding + checkpoint resume |
| AMP training | Fulfilled | autocast + GradScaler |
| Early stopping | Fulfilled | Patience-based on tampered Dice |


## 1. Environment Setup

Installs dependencies, defines dataset paths, and locates the dataset using a three-tier
fallback: Kaggle attached dataset → Google Drive → Kaggle API download.

In [1]:
import subprocess
import sys
import os
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
KAGGLE_DATASET_SLUG = "harshv777/casia2-0-upgraded-dataset"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")
ATTACHED_DATASET_DIR = KAGGLE_INPUT_DIR / "casia2-0-upgraded-dataset"
DRIVE_SEARCH_ROOTS = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "albumentations==1.3.1", "opencv-python-headless==4.10.0.84", "kaggle",
    ])

KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("KAGGLE_INPUT_DIR:", KAGGLE_INPUT_DIR)
print("KAGGLE_WORKING_DIR:", KAGGLE_WORKING_DIR)
print("ATTACHED_DATASET_DIR:", ATTACHED_DATASET_DIR)


IN_COLAB: False
KAGGLE_INPUT_DIR: /kaggle/input
KAGGLE_WORKING_DIR: /kaggle/working
ATTACHED_DATASET_DIR: /kaggle/input/casia2-0-upgraded-dataset


In [2]:
def has_valid_layout(path):
    """Check for IMAGE/ and MASK/ subdirectories."""
    if not path or not path.is_dir():
        return False
    children = {c.name.lower() for c in path.iterdir() if c.is_dir()}
    return "image" in children and "mask" in children


def find_in_kaggle_input(input_dir):
    """Search /kaggle/input/ for any attached dataset with IMAGE+MASK layout."""
    if input_dir is None or not input_dir.exists() or not input_dir.is_dir():
        return None
    preferred, fallback = [], []
    # Exact slug match
    exact = input_dir / "casia2-0-upgraded-dataset"
    if exact.exists() and has_valid_layout(exact):
        preferred.append(exact)
    # Direct children
    for child in sorted(input_dir.iterdir()):
        if child.is_dir() and has_valid_layout(child):
            fallback.append(child)
    # Recursive scan (Kaggle may nest under /kaggle/input/datasets/user/slug/)
    for d in sorted(input_dir.rglob("*")):
        if d.is_dir() and has_valid_layout(d):
            if "casia" in d.name.lower():
                preferred.append(d)
            else:
                fallback.append(d)
    candidates = preferred or fallback
    return candidates[0] if candidates else None


def find_in_drive(roots):
    """Search Google Drive mount points for the dataset folder."""
    for root in roots:
        if not root.exists():
            continue
        candidate = root / "casia2-0-upgraded-dataset"
        if candidate.exists() and has_valid_layout(candidate):
            return candidate
        for match in root.rglob("*casia*"):
            if match.is_dir() and has_valid_layout(match):
                return match
    return None


def download_from_kaggle(slug, target_dir):
    """Download dataset via Kaggle API to /kaggle/working/ (writable)."""
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        # Download to writable directory, NOT /kaggle/input/ (read-only on Kaggle)
        download_dir = KAGGLE_WORKING_DIR / "_dataset_download"
        download_dir.mkdir(parents=True, exist_ok=True)
        api.dataset_download_files(slug, path=str(download_dir), unzip=True, quiet=False)
        if has_valid_layout(download_dir):
            return download_dir
        for d in download_dir.rglob("*"):
            if d.is_dir() and has_valid_layout(d):
                return d
    except Exception as e:
        print(f"Kaggle API fallback failed: {e}")
    return None


In [3]:
# DEBUG: show what Kaggle attached under /kaggle/input/
if KAGGLE_INPUT_DIR.exists():
    print("Contents of", KAGGLE_INPUT_DIR)
    for p in sorted(KAGGLE_INPUT_DIR.iterdir()):
        print(f"  {p.name}/  ->  {[c.name for c in p.iterdir() if c.is_dir()][:10] if p.is_dir() else 'file'}")
else:
    print(f"{KAGGLE_INPUT_DIR} does not exist")

dataset_dir = None

# 1) Kaggle attached dataset (default) — scan all attached datasets
dataset_dir = find_in_kaggle_input(KAGGLE_INPUT_DIR)
if dataset_dir:
    print(f"Using attached Kaggle dataset: {dataset_dir}")

# 2) Google Drive fallback (Colab only)
if dataset_dir is None and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        found = find_in_drive(DRIVE_SEARCH_ROOTS)
        if found:
            dataset_dir = found
            print(f"Using Google Drive dataset: {dataset_dir}")
    except Exception as e:
        print(f"Google Drive fallback skipped: {e}")

# 3) Kaggle API download (last resort — writes to /kaggle/working/)
if dataset_dir is None:
    dataset_dir = download_from_kaggle(KAGGLE_DATASET_SLUG, KAGGLE_WORKING_DIR)
    if dataset_dir:
        print(f"Downloaded dataset via Kaggle API: {dataset_dir}")

if not dataset_dir or not has_valid_layout(dataset_dir):
    raise FileNotFoundError(
        "Dataset not found. Attach harshv777/casia2-0-upgraded-dataset on Kaggle, "
        "or place it in Google Drive, or configure Kaggle API credentials."
    )

print(f"Dataset root: {dataset_dir}")
print(f"  IMAGE dir: {(dataset_dir / 'IMAGE').exists()}")
print(f"  MASK dir:  {(dataset_dir / 'MASK').exists()}")


Contents of /kaggle/input
  datasets/  ->  ['harshv777']
Using attached Kaggle dataset: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset
Dataset root: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset
  IMAGE dir: True
  MASK dir:  True


In [4]:
import os
devnull = os.open(os.devnull, os.O_WRONLY)
os.dup2(devnull, 2)  # redirect C-level stderr to /dev/null

2

## 2. Configuration

All hyperparameters, feature flags, and path settings are centralized in a single
`CONFIG` dictionary.  This replaces the scattered constants from previous notebook
versions and makes experiment iteration easier.  Every training, evaluation, and
data-loading cell reads from `CONFIG` instead of defining its own local constants.


In [5]:
import os

SEED = 42

CONFIG = {
    # -- Data --
    'image_size': 256,
    'batch_size': 8,             # auto-adjusted based on GPU VRAM
    'num_workers': 4,
    'train_ratio': 0.70,

    # -- Model --
    'architecture': 'UNetWithClassifier',
    'n_channels': 3,
    'n_classes': 1,
    'n_labels': 2,
    'dropout': 0.5,

    # -- Optimizer --
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'max_grad_norm': 5.0,

    # -- Scheduler --
    'scheduler': 'CosineAnnealingLR',
    'scheduler_T_max': 50,            # single cosine decay over all epochs

    # -- Loss --
    'alpha': 1.5,                # classification loss weight
    'beta': 1.0,                 # segmentation loss weight
    'focal_gamma': 2.0,
    'seg_bce_weight': 0.5,
    'seg_dice_weight': 0.5,

    # -- Training --
    'max_epochs': 100,
    'patience': 50,              # early stopping patience
    'checkpoint_every': 10,      # periodic checkpoint interval

    # -- Feature Flags --
    'use_amp': True,
    'use_wandb': True,
    'seg_threshold': 0.5,

    # -- Reproducibility --
    'seed': SEED,
}

# -- Output Directories --
CHECKPOINT_DIR = os.path.join(str(KAGGLE_WORKING_DIR), 'checkpoints')
RESULTS_DIR    = os.path.join(str(KAGGLE_WORKING_DIR), 'results')
PLOTS_DIR      = os.path.join(str(KAGGLE_WORKING_DIR), 'plots')

for d in [CHECKPOINT_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'\nCHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'RESULTS_DIR:    {RESULTS_DIR}')
print(f'PLOTS_DIR:      {PLOTS_DIR}')

CONFIG:
  image_size: 256
  batch_size: 8
  num_workers: 4
  train_ratio: 0.7
  architecture: UNetWithClassifier
  n_channels: 3
  n_classes: 1
  n_labels: 2
  dropout: 0.5
  learning_rate: 0.0001
  weight_decay: 0.0001
  max_grad_norm: 5.0
  scheduler: CosineAnnealingLR
  scheduler_T_max: 50
  alpha: 1.5
  beta: 1.0
  focal_gamma: 2.0
  seg_bce_weight: 0.5
  seg_dice_weight: 0.5
  max_epochs: 100
  patience: 50
  checkpoint_every: 10
  use_amp: True
  use_wandb: True
  seg_threshold: 0.5
  seed: 42

CHECKPOINT_DIR: /kaggle/working/checkpoints
RESULTS_DIR:    /kaggle/working/results
PLOTS_DIR:      /kaggle/working/plots


### 2.1 Hyperparameter Summary

| Hyperparameter | Value | Source |
|---|---|---|
| `image_size` | 256 | Preserved from vK.7.1 |
| `batch_size` | 8 (auto-adjusted) | VRAM-based |
| `learning_rate` | 1e-4 | Preserved |
| `max_epochs` | 50 | Preserved |
| `alpha` (cls weight) | 1.5 | Preserved |
| `beta` (seg weight) | 1.0 | Preserved |
| `focal_gamma` | 2.0 | Preserved |
| `scheduler` | CosineAnnealingLR(T_max=10) | Preserved |
| `patience` | 10 | New (early stopping) |
| `use_amp` | True | New (mixed precision) |


## 3. Reproducibility and Device Setup

Full reproducibility is enforced through seeded random number generators across
Python, NumPy, and PyTorch.  GPU diagnostics confirm hardware capabilities and
auto-adjust the batch size based on available VRAM.


In [6]:
import random
import numpy as np
import torch


def set_seed(seed):
    """Set seeds for full reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    n_gpus = torch.cuda.device_count()

    # TF32 for faster matmul on Ampere+ (does not affect determinism)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print(f'GPU:            {gpu_name}')
    print(f'VRAM:           {vram_gb:.1f} GB')
    print(f'GPUs available: {n_gpus}')
    print(f'cuDNN:          {torch.backends.cudnn.enabled}')
    print(f'Deterministic:  {torch.backends.cudnn.deterministic}')
    print(f'AMP:            {"enabled" if CONFIG["use_amp"] else "disabled"}')

    # Auto-adjust batch size based on available VRAM
    if vram_gb >= 15:
        CONFIG['batch_size'] = 32
    elif vram_gb >= 10:
        CONFIG['batch_size'] = 16
    # else keep CONFIG default (8)
    print(f'Batch size (auto-adjusted): {CONFIG["batch_size"]}')
else:
    print('WARNING: No GPU detected. Training will be extremely slow.')

print(f'Device: {device}')

GPU:            Tesla T4
VRAM:           15.6 GB
GPUs available: 2
cuDNN:          True
Deterministic:  True
AMP:            enabled
Batch size (auto-adjusted): 32
Device: cuda


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


The batch size is auto-adjusted based on available GPU VRAM:
- **>=15 GB** (Kaggle P100 / Colab T4): `batch_size=32`
- **>=10 GB**: `batch_size=16`
- **<10 GB**: `batch_size=8` (default)

The effective hyperparameters logged to W&B reflect this adjusted value.


## 4. Dataset Discovery and Metadata Cache

The notebook operates on an image tampering dataset organized into authentic and
tampered images with corresponding binary ground truth masks.

**Metadata caching:** If a valid `image_mask_metadata.csv` already exists with the
expected row count, the scanning step is skipped entirely to reduce startup time.


#### 2.1.1 Locate IMAGE and MASK Directories

Scans the normalized Kaggle-style input path and searches recursively for the `IMAGE` and `MASK` folders.

In [7]:
import os
from pathlib import Path
import pandas as pd

# =========================
# 1) Discover the dataset root
# =========================

# Inspect the Kaggle input directory so the notebook can confirm the expected dataset is available.
INPUT_ROOT = KAGGLE_INPUT_DIR

print(f"Available datasets under {INPUT_ROOT}:")
for p in INPUT_ROOT.iterdir():
    if p.is_dir():
        print(" -", p.name)

# Use the Kaggle-first attached dataset path prepared in the environment setup cells.
DATASET_DIR = dataset_dir  # resolved in Environment Setup (Kaggle -> Drive -> API)

# Search recursively for IMAGE and MASK so the metadata build still works if the dataset is nested one level deeper.
IMAGE_DIR = None
MASK_DIR = None

for p in DATASET_DIR.rglob("*"):
    if p.is_dir() and p.name.lower() == "image":
        IMAGE_DIR = p
    if p.is_dir() and p.name.lower() == "mask":
        MASK_DIR = p

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)

assert IMAGE_DIR is not None, "Could not find the IMAGE directory. Verify the dataset folder name and structure."
assert MASK_DIR is not None, "Could not find the MASK directory. Verify the dataset folder name and structure."

Available datasets under /kaggle/input:
 - datasets
IMAGE_DIR: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE
MASK_DIR: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/MASK


#### 2.1.2 Build Metadata Table from Image-Mask Pairs

Iterates over authentic (`Au`) and tampered (`Tp`) subdirectories,
pairing each image with its corresponding mask and recording the image-level label.


In [8]:
# =========================
# Build metadata with caching
# =========================

output_csv = KAGGLE_WORKING_DIR / "image_mask_metadata.csv"

# Count images in dataset to validate cache
_au_count = sum(1 for f in (IMAGE_DIR / 'Au').iterdir() if f.is_file())
_tp_count = sum(1 for f in (IMAGE_DIR / 'Tp').iterdir() if f.is_file())
_expected_count = _au_count + _tp_count

df = None
if output_csv.exists():
    cached_df = pd.read_csv(output_csv)
    if len(cached_df) == _expected_count:
        print(f'Using cached metadata ({len(cached_df)} rows): {output_csv}')
        df = cached_df
    else:
        print(f'Cache stale ({len(cached_df)} rows vs {_expected_count} expected), rebuilding...')

if df is None:
    label_folders = {
        "Au": {"class_name": "authentic", "label": 0},
        "Tp": {"class_name": "tampered",  "label": 1},
    }
    valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
    rows = []

    for sub_name, info in label_folders.items():
        img_subdir = IMAGE_DIR / sub_name
        mask_subdir = MASK_DIR / sub_name
        if not img_subdir.exists():
            print(f'Warning: image folder does not exist: {img_subdir}')
            continue
        print(f'Scanning images in: {img_subdir}')
        for img_path in img_subdir.iterdir():
            if not img_path.is_file():
                continue
            if img_path.suffix.lower() not in valid_exts:
                continue
            mask_path = mask_subdir / img_path.name
            mask_exists = mask_path.exists()
            rows.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path) if mask_exists else "",
                "mask_exists": int(mask_exists),
                "class_folder": sub_name,
                "class_name": info["class_name"],
                "label": info["label"],
            })

    print(f'Total images found: {len(rows)}')
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'CSV saved to: {output_csv}')

print(df.head())
print('\nCounts per class_name:')
print(df['class_name'].value_counts())
print('\nMissing masks:')
print(df[df['mask_exists'] == 0].head())

Scanning images in: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE/Au
Scanning images in: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE/Tp
Total images found: 12614
CSV saved to: /kaggle/working/image_mask_metadata.csv
                                          image_path  \
0  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
1  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
2  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
3  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
4  /kaggle/input/datasets/harshv777/casia2-0-upgr...   

                                           mask_path  mask_exists  \
0  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
1  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
2  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
3  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
4  /kaggle/input/datasets/harshv777/casia2-0-upgr...          

### 2.2 Train / Validation / Test Split

Loads the metadata CSV, applies a stratified 70/15/15 split to preserve class balance
across train, validation, and test subsets, and saves each split to a separate CSV file.


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_csv = KAGGLE_WORKING_DIR / 'train_metadata.csv'
val_csv   = KAGGLE_WORKING_DIR / 'val_metadata.csv'
test_csv  = KAGGLE_WORKING_DIR / 'test_metadata.csv'

# Check for cached splits
if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv)
    test_df  = pd.read_csv(test_csv)
    if len(train_df) + len(val_df) + len(test_df) == len(df):
        print(f'Using cached splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    else:
        print('Cached splits stale, rebuilding...')
        train_df = None
else:
    train_df = None

if train_df is None:
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df['label'], random_state=CONFIG['seed'],
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['label'], random_state=CONFIG['seed'],
    )
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    print(f'Splits saved: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')

print(f'\nTrain class distribution:')
print(train_df['class_name'].value_counts())
print(f'\nVal class distribution:')
print(val_df['class_name'].value_counts())
print(f'\nTest class distribution:')
print(test_df['class_name'].value_counts())

Splits saved: train=8829, val=1892, test=1893

Train class distribution:
class_name
authentic    5243
tampered     3586
Name: count, dtype: int64

Val class distribution:
class_name
authentic    1124
tampered      768
Name: count, dtype: int64

Test class distribution:
class_name
authentic    1124
tampered      769
Name: count, dtype: int64


### 4.4 Dataset Summary

Quick summary of dataset splits and class balance.

In [10]:
print(f"{'Split':<8} {'Total':>6} {'Authentic':>10} {'Tampered':>9}")
print('-' * 38)
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_auth = (split_df['label'] == 0).sum()
    n_tamp = (split_df['label'] == 1).sum()
    print(f'{name:<8} {len(split_df):>6} {n_auth:>10} {n_tamp:>9}')

Split     Total  Authentic  Tampered
--------------------------------------
Train      8829       5243      3586
Val        1892       1124       768
Test       1893       1124       769


## 5. Dependencies and Imports

All training, evaluation, and visualization imports are consolidated here
to avoid redundant imports scattered across multiple cells.


In [11]:
import cv2
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt

print('Imports complete.')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Imports complete.
PyTorch: 2.9.0+cu126
CUDA available: True


## 6. Data Loading and Preprocessing

The source notebook loads metadata CSV files, builds PyTorch datasets with
Albumentations augmentation, and creates dataloaders for training, validation,
and test splits.


### 6.1 Image and Mask Transforms

Defines Albumentations pipelines for the training split (with augmentation) and the
validation/test splits (resize and normalization only).


In [12]:
# ================== Define transforms ==================
IMAGE_SIZE = CONFIG['image_size']

def get_train_transform():
    """
    Purpose:
        Build the augmentation pipeline used for training images and masks.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.25),
        A.ImageCompression(quality_lower=50, quality_upper=90, p=0.25),
        A.ShiftScaleRotate(
            shift_limit=0.02,
            scale_limit=0.1,
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5,
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_valid_transform():
    """
    Purpose:
        Build the deterministic preprocessing pipeline for validation and test samples.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

### 6.2 Dataset Class

The `ImageMaskDataset` class loads image-mask pairs from metadata, applies shared transforms,
and returns tensors compatible with the dual-head model.


In [13]:
# ================== 3) Define the dataset ==================
class ImageMaskDataset(Dataset):
    """
    Purpose:
        Load paired images, masks, and image-level tamper labels from the metadata CSV files.

    Key Responsibilities:
        - Read each image and its corresponding mask from disk.
        - Apply shared preprocessing so classification and localization are trained on aligned inputs.

    Notes:
        The dataset returns image tensor, binary mask tensor, and image-level label.
    """
    def __init__(self, df, transform=None):
        """
        Purpose:
            Store metadata and an optional transform pipeline for later sample loading.

        Inputs:
            df (pd.DataFrame): Metadata table with image paths, mask paths, and labels.
            transform (callable): Optional Albumentations transform applied to image and mask.

        Returns:
            None.

        Notes:
            The dataframe index is reset to preserve stable integer access during training.
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        """
        Purpose:
            Return the number of examples in the dataset.

        Inputs:
            None.

        Returns:
            int: Number of metadata rows.

        Notes:
            DataLoader uses this length to define epoch size.
        """
        return len(self.df)

    def __getitem__(self, idx):
        """
        Purpose:
            Load one image, its binary mask, and the image-level tamper label.

        Inputs:
            idx (int): Sample index inside the metadata table.

        Returns:
            tuple: Image tensor, mask tensor, and class label tensor.

        Notes:
            Binary masks make the segmentation target explicit: tampered pixels vs clean pixels.
        """
        row = self.df.iloc[idx]

        img_path = row["image_path"]
        mask_path = row["mask_path"]
        label = int(row["label"])

        image = cv2.imread(img_path)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        if mask is None:
            # Authentic images may lack a physical mask file â use zeros
            mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

        # Convert the OpenCV image to RGB before normalization and visualization.
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Convert the grayscale mask to binary so the localization target has a clear foreground/background split.
        mask = (mask > 0).astype("float32")

        if self.transform is not None:
            # Apply identical spatial augmentation to the image and its supervision mask.
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask  = torch.from_numpy(mask).unsqueeze(0).float()

        if mask.ndim == 2:
            # Keep an explicit channel dimension for the segmentation branch and BCE-style losses.
            mask = mask.unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)


### 6.3 DataLoader Construction

Creates train, validation, and test DataLoaders with optimized settings:
- `persistent_workers` to avoid respawning workers each epoch
- `seed_worker` + `Generator` for reproducible data ordering
- `drop_last=True` for training to avoid uneven last-batch size
- `pin_memory=True` for faster GPU transfers


In [14]:
def seed_worker(worker_id):
    """Seed each DataLoader worker for reproducibility."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(CONFIG['seed'])

_nw = CONFIG['num_workers']
loader_kwargs = dict(
    num_workers=_nw,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=_nw > 0,
)

train_dataset = ImageMaskDataset(train_df, transform=get_train_transform())
val_dataset   = ImageMaskDataset(val_df,   transform=get_valid_transform())
test_dataset  = ImageMaskDataset(test_df,  transform=get_valid_transform())

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True,
    worker_init_fn=seed_worker,
    generator=g,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test:  {len(test_dataset)} samples, {len(test_loader)} batches')
print(f'Workers: {_nw}, persistent: {loader_kwargs["persistent_workers"]}, batch_size: {CONFIG["batch_size"]}')

Train: 8829 samples, 275 batches
Val:   1892 samples, 60 batches
Test:  1893 samples, 60 batches
Workers: 4, persistent: True, batch_size: 32


/tmp/ipykernel_24/1839044940.py:14: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=50, quality_upper=90, p=0.25),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## 7. Model Architecture

The implemented model is a custom U-Net-style encoder-decoder with a classifier head.

- `DoubleConv`: two conv-BN-ReLU blocks
- `Down`: max-pool + DoubleConv (encoder stage)
- `Up`: transposed conv + skip concatenation + DoubleConv (decoder stage)
- `UNetWithClassifier`: shared backbone with segmentation output and classification head

**Note:** The architecture is preserved exactly from the source notebook.


In [15]:
# ================== 4) Define the U-Net with classifier head ==================
class DoubleConv(nn.Module):
    """
    Purpose:
        Apply two convolutional layers with normalization and activation.

    Key Responsibilities:
        - Extract localized visual features.
        - Reuse the same feature block across encoder and decoder stages.

    Notes:
        This block is the core building unit of the U-Net backbone.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize the double-convolution block.

        Inputs:
            in_ch (int): Number of input channels.
            out_ch (int): Number of output channels.

        Returns:
            None.

        Notes:
            Batch normalization follows each convolution.
        """
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        """
        Purpose:
            Transform the input feature map with two convolution stages.

        Inputs:
            x (torch.Tensor): Input feature tensor.

        Returns:
            torch.Tensor: Feature tensor after convolution, normalization, and activation.

        Notes:
            This helper keeps encoder and decoder definitions concise.
        """
        return self.block(x)

class Down(nn.Module):
    """
    Purpose:
        Downsample feature maps in the encoder path.

    Key Responsibilities:
        - Reduce spatial resolution with max pooling.
        - Expand channel capacity with a DoubleConv block.

    Notes:
        This block helps the network capture higher-level tampering cues.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize one encoder stage.

        Inputs:
            in_ch (int): Number of incoming channels.
            out_ch (int): Number of outgoing channels.

        Returns:
            None.

        Notes:
            Pooling is applied before the convolution block.
        """
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        """
        Purpose:
            Downsample the input feature map and refine it.

        Inputs:
            x (torch.Tensor): Encoder input feature map.

        Returns:
            torch.Tensor: Downsampled and refined feature map.

        Notes:
            The sequence preserves the original implementation exactly.
        """
        x = self.pool(x)
        x = self.conv(x)
        return x

class Up(nn.Module):
    """
    Purpose:
        Upsample decoder features and fuse them with encoder skip connections.

    Key Responsibilities:
        - Recover spatial detail in the decoder.
        - Merge coarse semantic features with fine-grained encoder activations.

    Notes:
        Padding compensates for minor shape mismatches before concatenation.
    """
    def __init__(self, in_channels, out_channels):
        """
        Purpose:
            Initialize one decoder stage.

        Inputs:
            in_channels (int): Number of channels entering the transposed convolution.
            out_channels (int): Number of channels produced after upsampling.

        Returns:
            None.

        Notes:
            The concatenated tensor is refined with DoubleConv.
        """
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        """
        Purpose:
            Upsample decoder features and concatenate them with encoder skip features.

        Inputs:
            x1 (torch.Tensor): Decoder feature map from the deeper level.
            x2 (torch.Tensor): Encoder skip connection from the matching spatial scale.

        Returns:
            torch.Tensor: Decoder feature map after skip fusion.

        Notes:
            Shape alignment is handled with explicit padding before concatenation.
        """
        x1 = self.up(x1)

        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNetWithClassifier(nn.Module):
    """
    Purpose:
        Predict both image-level tampering labels and pixel-level tampering masks.

    Key Responsibilities:
        - Use a shared encoder-decoder backbone for localization.
        - Use a classifier head on the bottleneck to detect whether an image is tampered.

    Notes:
        The shared backbone lets the model learn complementary detection and localization signals.
    """
    def __init__(self, n_channels=3, n_classes=1, n_labels=2):
        """
        Purpose:
            Construct the shared U-Net backbone and the auxiliary classifier head.

        Inputs:
            n_channels (int): Number of image channels.
            n_classes (int): Number of segmentation output channels.
            n_labels (int): Number of classification labels.

        Returns:
            None.

        Notes:
            The architecture is preserved from the original notebook without structural changes.
        """
        super().__init__()

        # Encoder
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        # Decoder
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_labels),
        )

    def forward(self, x):
        """
        Purpose:
            Produce image-level logits and segmentation logits from the same input batch.

        Inputs:
            x (torch.Tensor): Batch of input images.

        Returns:
            tuple: Classification logits and segmentation logits.

        Notes:
            Classification uses the bottleneck representation, while segmentation uses the decoder output.
        """
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)   # bottleneck

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)
        seg_logits = self.outc(x)

        cls_logits = self.classifier(x5)

        return cls_logits, seg_logits


In [16]:
model = UNetWithClassifier(
    n_channels=CONFIG['n_channels'],
    n_classes=CONFIG['n_classes'],
    n_labels=CONFIG['n_labels'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

Total parameters:     31,563,459
Trainable parameters: 31,563,459


## 8. Experiment Tracking

W&B is attached for experiment tracking.  Controlled by `CONFIG['use_wandb']`.
Falls back to offline mode if Kaggle secrets are unavailable.


In [17]:
import importlib.util
import subprocess

WANDB_ACTIVE = False
WANDB_RUN = None

if CONFIG['use_wandb']:
    WANDB_CONFIG = {k: v for k, v in CONFIG.items()}

    try:
        if importlib.util.find_spec("wandb") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])

        import wandb

        try:
            from kaggle_secrets import UserSecretsClient
            wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
            if not wandb_api_key:
                raise ValueError('WANDB_API_KEY is empty')
            wandb.login(key=wandb_api_key)
            WANDB_RUN = wandb.init(
                project="vK.10.3-tampered-image-detection-assignment",
                name=f"vK.10.3-unet-seed{SEED}-kaggle",
                tags=["vk10.3", "assignment", "amp", "checkpointing", "early-stopping"],
                config=WANDB_CONFIG,
                reinit=True,
            )
            WANDB_ACTIVE = True
        except Exception as auth_exc:
            print(f"W&B online unavailable, switching to offline: {auth_exc}")
            WANDB_RUN = wandb.init(
                project="tampered-image-detection-assignment",
                name="vk10.3-offline",
                config=WANDB_CONFIG,
                mode="offline",
                reinit=True,
            )
            WANDB_ACTIVE = True
    except Exception as exc:
        print(f"W&B setup failed: {exc}")

print(f"W&B active: {WANDB_ACTIVE}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harsh_vardhan (harsh_vardhan_iiitn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run x2wtjku6
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260313_231820-x2wtjku6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vK.10.3-unet-seed42-kaggle
wandb: ⭐️ View project at https://wandb.ai/harsh_vardhan_iiitn/vK.10.3-t

W&B active: True


## 9. Training Utilities

This section defines loss functions, evaluation metrics, checkpoint helpers,
and initializes the AMP scaler.  Loss functions are preserved verbatim from
the source notebook.


In [18]:
# ================== Loss functions, optimizer, scheduler, AMP scaler ==================

# Compute class weights from the training split
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)


class FocalLoss(nn.Module):
    """Focal-style classification loss that down-weights easy examples."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def dice_loss(pred, target, eps=1e-7):
    """Soft Dice loss for segmentation logits."""
    prob = torch.sigmoid(pred)
    inter = (prob * target).sum(dim=(1,2,3))
    union = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return 1 - dice.mean()


criterion_cls = FocalLoss(alpha=class_weights, gamma=CONFIG['focal_gamma'])
bce_loss      = nn.BCEWithLogitsLoss()
ALPHA = CONFIG['alpha']
BETA  = CONFIG['beta']

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['scheduler_T_max'])

# AMP scaler
scaler = GradScaler('cuda', enabled=CONFIG['use_amp'])

print(f'Optimizer: Adam(lr={CONFIG["learning_rate"]}, wd={CONFIG["weight_decay"]})')
print(f'Scheduler: CosineAnnealingLR(T_max={CONFIG["scheduler_T_max"]})')
print(f'AMP: {"enabled" if CONFIG["use_amp"] else "disabled"}')
print(f'Loss weights: alpha={ALPHA}, beta={BETA}')

Class weights: tensor([0.8420, 1.2310], device='cuda:0')
Optimizer: Adam(lr=0.0001, wd=0.0001)
Scheduler: CosineAnnealingLR(T_max=50)
AMP: enabled
Loss weights: alpha=1.5, beta=1.0


### 9.1 Evaluation Metrics

Dice, IoU, and F1 are computed on thresholded binary masks.  To address metric
inflation from authentic images (where both prediction and ground truth are empty),
`compute_metrics_split()` reports metrics **separately for tampered-only samples**.


In [19]:
# ================== Evaluation Metrics ==================

def dice_coef(pred, target, eps=1e-7):
    """Dice coefficient for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return dice.mean().item()


def iou_coef(pred, target, eps=1e-7):
    """IoU for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()


def f1_coef(pred, target, eps=1e-7):
    """Pixel-level F1 for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    tp = (pred_bin * target).sum(dim=(1,2,3))
    fp = (pred_bin * (1.0 - target)).sum(dim=(1,2,3))
    fn = ((1.0 - pred_bin) * target).sum(dim=(1,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return (2.0 * precision * recall / (precision + recall + eps)).mean().item()


def compute_metrics_split(seg_logits, masks, labels):
    """Compute metrics for all samples and tampered-only samples separately."""
    all_dice, all_iou, all_f1 = [], [], []
    tam_dice, tam_iou, tam_f1 = [], [], []

    for i in range(seg_logits.size(0)):
        sl = seg_logits[i:i+1]
        m  = masks[i:i+1]
        d  = dice_coef(sl, m)
        io = iou_coef(sl, m)
        f  = f1_coef(sl, m)
        all_dice.append(d)
        all_iou.append(io)
        all_f1.append(f)

        if labels[i].item() == 1:  # tampered only
            tam_dice.append(d)
            tam_iou.append(io)
            tam_f1.append(f)

    return {
        'dice': np.mean(all_dice),
        'iou': np.mean(all_iou),
        'f1': np.mean(all_f1),
        'tampered_dice': np.mean(tam_dice) if tam_dice else 0.0,
        'tampered_iou': np.mean(tam_iou) if tam_iou else 0.0,
        'tampered_f1': np.mean(tam_f1) if tam_f1 else 0.0,
    }

### 9.2 Checkpoint Helpers

Three-file checkpoint strategy:
- `last_checkpoint.pt` — saved every epoch for seamless resume
- `best_model.pt` — saved when tampered-only Dice improves
- `checkpoint_epoch_N.pt` — periodic snapshot every N epochs


In [20]:
def save_checkpoint(state, filepath):
    """Save training state to a checkpoint file."""
    torch.save(state, filepath)


def load_checkpoint(filepath, model, optimizer, scaler, scheduler=None):
    """Restore training state from a checkpoint file."""
    ckpt = torch.load(filepath, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    restored_history = ckpt.get('history', None)
    return ckpt['epoch'] + 1, ckpt.get('best_metric', 0.0), ckpt.get('best_epoch', 0), restored_history


print('Checkpoint helpers defined.')


Checkpoint helpers defined.


## 10. Training Loop

The training loop uses:
- **AMP** for mixed precision training
- **Gradient clipping** at `max_grad_norm` for stability
- **Three-file checkpointing** with automatic resume
- **Early stopping** based on tampered-only Dice coefficient
- **Tampered-only metric tracking** for honest evaluation


In [21]:
def train_one_epoch(epoch):
    """Train for one epoch with AMP and gradient clipping. Returns loss, acc, and train dice."""
    model.train()
    running_loss = 0.0
    correct = 0
    all_seg_logits, all_masks, all_labels = [], [], []

    for images, masks, labels in train_loader:
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = CONFIG['seg_bce_weight'] * bce_loss(seg_logits, masks) + \
                       CONFIG['seg_dice_weight'] * dice_loss(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['max_grad_norm'])
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()

        all_seg_logits.append(seg_logits.detach().cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())

    scheduler.step()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / len(train_dataset)

    # Compute train segmentation dice (tampered-only)
    all_seg_logits = torch.cat(all_seg_logits)
    all_masks = torch.cat(all_masks)
    all_labels = torch.cat(all_labels)
    tampered_mask = all_labels == 1
    if tampered_mask.any():
        train_dice = dice_coef(all_seg_logits[tampered_mask], all_masks[tampered_mask])
    else:
        train_dice = 0.0

    return epoch_loss, epoch_acc, train_dice


In [22]:
@torch.no_grad()
def evaluate(loader, dataset_len, name='Val'):
    """Evaluate model with AMP, returning all-sample and tampered-only metrics."""
    model.eval()
    running_loss = 0.0
    correct = 0
    all_cls_logits, all_seg_logits, all_masks, all_labels = [], [], [], []

    for images, masks, labels in loader:
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = CONFIG['seg_bce_weight'] * bce_loss(seg_logits, masks) +                        CONFIG['seg_dice_weight'] * dice_loss(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()

        all_cls_logits.append(cls_logits.cpu())
        all_seg_logits.append(seg_logits.cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())

    all_cls_logits = torch.cat(all_cls_logits)
    all_seg_logits = torch.cat(all_seg_logits)
    all_masks = torch.cat(all_masks)
    all_labels = torch.cat(all_labels)

    seg_metrics = compute_metrics_split(all_seg_logits, all_masks, all_labels)

    epoch_loss = running_loss / dataset_len
    epoch_acc = correct / dataset_len

    # ROC-AUC for classification
    probs = torch.softmax(all_cls_logits, dim=1)[:, 1].numpy()
    labels_np = all_labels.numpy()
    try:
        auc = roc_auc_score(labels_np, probs)
    except ValueError:
        auc = 0.0  # single class in batch

    seg_metrics['loss'] = epoch_loss
    seg_metrics['acc'] = epoch_acc
    seg_metrics['roc_auc'] = auc

    print(
        f'  {name} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | AUC: {auc:.4f} | '
        f'Dice(tam): {seg_metrics["tampered_dice"]:.4f} | '
        f'IoU(tam): {seg_metrics["tampered_iou"]:.4f}'
    )
    return seg_metrics


In [23]:
# ================== Training state initialization ==================
history = {
    "train_loss": [], "train_acc": [], "train_dice": [],
    "val_loss": [], "val_acc": [],
    "val_dice": [], "val_iou": [], "val_f1": [],
    "val_tampered_dice": [], "val_tampered_iou": [], "val_tampered_f1": [],
    "val_roc_auc": [],
    "lr": [],
}

best_metric = 0.0  # tampered-only Dice
best_epoch = 0
start_epoch = 1

# Resume from checkpoint if available
resume_path = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')
if os.path.exists(resume_path):
    start_epoch, best_metric, best_epoch, restored_history = load_checkpoint(
        resume_path, model, optimizer, scaler, scheduler
    )
    if restored_history:
        history = restored_history
    print(f'Resumed from epoch {start_epoch}, best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
else:
    print('Starting fresh training.')


Starting fresh training.


In [24]:
# ================== Main training loop ==================
best_model_path = os.path.join(str(KAGGLE_WORKING_DIR), 'best_model.pth')

for epoch in range(start_epoch, CONFIG['max_epochs'] + 1):
    print(f'\nEpoch {epoch}/{CONFIG["max_epochs"]}')

    train_loss, train_acc, train_dice = train_one_epoch(epoch)
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Dice(tam): {train_dice:.4f}')

    val_metrics = evaluate(val_loader, len(val_dataset), name='Val')

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['acc'])
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_tampered_dice'].append(val_metrics['tampered_dice'])
    history['val_tampered_iou'].append(val_metrics['tampered_iou'])
    history['val_tampered_f1'].append(val_metrics['tampered_f1'])
    history['val_roc_auc'].append(val_metrics['roc_auc'])
    history['lr'].append(optimizer.param_groups[0]['lr'])

    # W&B logging
    if WANDB_ACTIVE:
        wandb.log({
            'epoch': epoch,
            'train/loss': train_loss, 'train/accuracy': train_acc, 'train/dice': train_dice,
            'val/loss': val_metrics['loss'], 'val/accuracy': val_metrics['acc'],
            'val/dice': val_metrics['dice'], 'val/iou': val_metrics['iou'],
            'val/f1': val_metrics['f1'],
            'val/tampered_dice': val_metrics['tampered_dice'],
            'val/tampered_iou': val_metrics['tampered_iou'],
            'val/tampered_f1': val_metrics['tampered_f1'],
            'val/roc_auc': val_metrics['roc_auc'],
            'lr': optimizer.param_groups[0]['lr'],
        })

    # Build checkpoint state (includes history for resume)
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'config': CONFIG,
        'history': history,
    }

    # Save last checkpoint every epoch
    save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt'))

    # Save history CSV every epoch for crash resilience
    pd.DataFrame(history).to_csv(os.path.join(RESULTS_DIR, 'training_history.csv'), index=False)

    # Best model selection: tampered-only Dice
    current_metric = val_metrics['tampered_dice']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        state['best_metric'] = best_metric
        state['best_epoch'] = best_epoch
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        torch.save(model.state_dict(), best_model_path)
        print(f'  => New best model (tampered Dice={best_metric:.4f})')

    # Periodic checkpoint
    if epoch % CONFIG['checkpoint_every'] == 0:
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pt'))

    # Early stopping
    if epoch - best_epoch >= CONFIG['patience']:
        print(f'Early stopping at epoch {epoch}. Best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
        break

print(f'\nTraining complete. Best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
print(f'Training history saved to {RESULTS_DIR}/training_history.csv')



Epoch 1/100
  Train Loss: 0.9351 | Train Acc: 0.4749 | Train Dice(tam): 0.0092
  Val Loss: 0.8777 | Acc: 0.4471 | AUC: 0.6478 | Dice(tam): 0.0012 | IoU(tam): 0.0008
  => New best model (tampered Dice=0.0012)

Epoch 2/100
  Train Loss: 0.8623 | Train Acc: 0.4636 | Train Dice(tam): 0.0019
  Val Loss: 0.8481 | Acc: 0.4244 | AUC: 0.6587 | Dice(tam): 0.0000 | IoU(tam): 0.0000

Epoch 3/100
  Train Loss: 0.8413 | Train Acc: 0.4870 | Train Dice(tam): 0.0010
  Val Loss: 0.8366 | Acc: 0.4477 | AUC: 0.6320 | Dice(tam): 0.0000 | IoU(tam): 0.0000

Epoch 4/100
  Train Loss: 0.8311 | Train Acc: 0.4933 | Train Dice(tam): 0.0004
  Val Loss: 0.8218 | Acc: 0.5433 | AUC: 0.6672 | Dice(tam): 0.0000 | IoU(tam): 0.0000

Epoch 5/100
  Train Loss: 0.8245 | Train Acc: 0.5191 | Train Dice(tam): 0.0009
  Val Loss: 0.8159 | Acc: 0.5534 | AUC: 0.6965 | Dice(tam): 0.0000 | IoU(tam): 0.0000

Epoch 6/100
  Train Loss: 0.8233 | Train Acc: 0.4990 | Train Dice(tam): 0.0003
  Val Loss: 0.8159 | Acc: 0.5613 | AUC: 0.6740 

## 11. Evaluation

Load the best checkpoint and evaluate on the held-out test split.
Reports both all-sample and tampered-only metrics.


In [25]:
# ================== Load best model and evaluate on test set ==================
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model from epoch {ckpt["best_epoch"]}')
else:
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print('Loaded best model from legacy path')

test_metrics = evaluate(test_loader, len(test_dataset), name='Test')

print(f'\n{"="*50}')
print('FINAL TEST RESULTS')
print(f'{"="*50}')
print(f'Accuracy:         {test_metrics["acc"]:.4f}')
print(f'Dice (all):       {test_metrics["dice"]:.4f}')
print(f'IoU (all):        {test_metrics["iou"]:.4f}')
print(f'F1 (all):         {test_metrics["f1"]:.4f}')
print(f'Dice (tampered):  {test_metrics["tampered_dice"]:.4f}')
print(f'IoU (tampered):   {test_metrics["tampered_iou"]:.4f}')
print(f'F1 (tampered):    {test_metrics["tampered_f1"]:.4f}')
print(f'ROC-AUC:          {test_metrics["roc_auc"]:.4f}')

TRAINING_HISTORY = history
FINAL_TEST_METRICS = test_metrics

if WANDB_ACTIVE:
    wandb.summary.update({
        'best_epoch': best_epoch,
        'test/accuracy': test_metrics['acc'],
        'test/dice': test_metrics['dice'],
        'test/tampered_dice': test_metrics['tampered_dice'],
        'test/tampered_iou': test_metrics['tampered_iou'],
        'test/tampered_f1': test_metrics['tampered_f1'],
        'test/roc_auc': test_metrics['roc_auc'],
    })

Loaded best model from epoch 96
  Test Loss: 0.6961 | Acc: 0.8304 | AUC: 0.8999 | Dice(tam): 0.2205 | IoU(tam): 0.1575

FINAL TEST RESULTS
Accuracy:         0.8304
Dice (all):       0.4298
IoU (all):        0.4042
F1 (all):         0.4298
Dice (tampered):  0.2205
IoU (tampered):   0.1575
F1 (tampered):    0.2205
ROC-AUC:          0.8999


### 11.1 Metric Inflation Note

**Why tampered-only metrics matter:** Authentic images have all-zero ground truth masks.
A model that predicts all-zeros on authentic images gets perfect Dice/IoU for those samples,
inflating aggregate scores.  The tampered-only metrics isolate localization quality on images
that actually contain manipulated regions.


### 11.2 Training Curves

In [26]:
history_df = pd.DataFrame(history) if history['train_loss'] else pd.read_csv(
    os.path.join(RESULTS_DIR, 'training_history.csv'))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

epochs_range = history_df.index + 1

# Loss curves
axes[0,0].plot(epochs_range, history_df['train_loss'], label='Train Loss')
axes[0,0].plot(epochs_range, history_df['val_loss'], label='Val Loss')
axes[0,0].set_title('Loss Curves')
axes[0,0].set_xlabel('Epoch')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Segmentation metrics (all + tampered-only)
if 'train_dice' in history_df.columns:
    axes[0,1].plot(epochs_range, history_df['train_dice'], label='Train Dice (tam)', ls='--', alpha=0.5)
axes[0,1].plot(epochs_range, history_df['val_dice'], label='Val Dice (all)', ls='--', alpha=0.5)
axes[0,1].plot(epochs_range, history_df['val_tampered_dice'], label='Dice (tampered)', lw=2)
axes[0,1].plot(epochs_range, history_df['val_tampered_iou'], label='IoU (tampered)')
axes[0,1].plot(epochs_range, history_df['val_tampered_f1'], label='F1 (tampered)')
axes[0,1].set_title('Segmentation Metrics')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Accuracy
axes[1,0].plot(epochs_range, history_df['train_acc'], label='Train Acc')
axes[1,0].plot(epochs_range, history_df['val_acc'], label='Val Acc')
axes[1,0].set_title('Image-Level Accuracy')
axes[1,0].set_xlabel('Epoch')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Learning rate
axes[1,1].plot(epochs_range, history_df['lr'], label='LR', color='orange')
axes[1,1].set_title('Learning Rate Schedule')
axes[1,1].set_xlabel('Epoch')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

if WANDB_ACTIVE:
    wandb.log({'training_curves': wandb.Image(fig)})

## 10. Visualization of Predictions

The following cells collect authentic and tampered examples from the test loader and visualize model outputs.
For assignment submission, the notebook makes the qualitative evidence clearer by surfacing:

- original image
- ground-truth mask
- predicted mask
- overlay highlighting predicted manipulated regions

Together, these views show both detection and localization performance.

**Assignment alignment:** This section provides qualitative evidence that the model detects tampering and highlights the manipulated region.


In [27]:
def denormalize(img_tensor):
    """
    Purpose:
        Convert a normalized image tensor back to displayable RGB space.

    Inputs:
        img_tensor (torch.Tensor): Normalized image tensor in CHW format.

    Returns:
        torch.Tensor: Image tensor scaled back to the [0, 1] range for visualization.

    Notes:
        This helper reverses the ImageNet normalization used during preprocessing.
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img

In [28]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print('Best model loaded for visualization.')

Best model loaded for visualization.


### 10.1 Sample Collection for Qualitative Review

The next helper functions collect balanced authentic and tampered examples from the test split and convert predictions into reviewer-friendly visualizations.
This makes it easier to compare image-level decisions with the corresponding predicted tampering regions.

**Assignment alignment:** This subsection connects the evaluation outputs to qualitative evidence for both detection and localization.


In [29]:
def collect_samples(loader, num_real=5, num_fake=5):
    """
    Purpose:
        Gather a balanced set of authentic and tampered examples for qualitative visualization.

    Inputs:
        loader (DataLoader): Dataloader that provides images, masks, and image-level labels.
        num_real (int): Number of authentic examples to collect.
        num_fake (int): Number of tampered examples to collect.

    Returns:
        tuple: Two lists containing authentic samples and tampered samples.

    Notes:
        Each sample stores the input image, ground-truth mask, predicted mask probabilities, and image-level labels.
    """
    real_samples = []
    fake_samples = []

    with torch.no_grad():
        for images, masks, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass: compute both image-level and pixel-level predictions for qualitative review.
            cls_logits, seg_logits = model(images)
            seg_probs = torch.sigmoid(seg_logits)
            preds = torch.argmax(cls_logits, dim=1)

            for i in range(images.size(0)):
                sample = {
                    "image": images[i].cpu(),
                    "mask_true": masks[i].cpu(),
                    "mask_pred": seg_probs[i].cpu(),
                    "label": int(labels[i].item()),
                    "pred": int(preds[i].item())
                }

                if sample["label"] == 0 and len(real_samples) < num_real:
                    real_samples.append(sample)

                if sample["label"] == 1 and len(fake_samples) < num_fake:
                    fake_samples.append(sample)

                if len(real_samples) >= num_real and len(fake_samples) >= num_fake:
                    return real_samples, fake_samples

    return real_samples, fake_samples


real_samples, fake_samples = collect_samples(test_loader, 5, 5)

print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

Collected Real: 5  Fake: 5


In [30]:
import matplotlib.pyplot as plt
import numpy as np

def show_samples_with_masks(samples, title):
    """
    Purpose:
        Visualize predicted tampered regions as red overlays on top of the original image.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the panel.

    Returns:
        None.

    Notes:
        Overlay views help the reader see where the model believes manipulation evidence is concentrated.
    """
    n = len(samples)
    plt.figure(figsize=(4*n, 4))

    for i, s in enumerate(samples):
        img = denormalize(s["image"]).permute(1,2,0).numpy()
        # Threshold the predicted probability map to produce a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Color predicted tampered pixels in red so the localization output is easy to interpret.
        overlay[mask_pred==1] = [1, 0, 0]

        blended = (0.6 * img + 0.4 * overlay)

        plt.subplot(1, n, i+1)
        plt.imshow(blended)
        lbl = "Real" if s["label"]==0 else "Fake"
        pred_lbl = "Real" if s["pred"]==0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_samples_with_masks(real_samples, "5 Real Images with Predicted Manipulation Regions")
show_samples_with_masks(fake_samples, "5 Fake Images with Predicted Manipulation Regions")

In [31]:
import matplotlib.pyplot as plt
import numpy as np

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display each image together with its predicted binary tampering mask.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample is shown in two rows: the original image first, then the thresholded predicted mask.
    """
    n = len(samples)
    plt.figure(figsize=(6*n, 6))

    for idx, s in enumerate(samples):
        # Convert the normalized tensor back to a displayable RGB image.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to obtain a black-and-white mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Top row: original image with image-level ground truth and prediction labels.
        plt.subplot(2, n, idx+1)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

        # Bottom row: predicted binary mask for tampered-region localization.
        plt.subplot(2, n, n + idx + 1)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Predicted Mask")
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [32]:
show_image_and_mask(real_samples, "5 Real Images + Predicted Masks")
show_image_and_mask(fake_samples, "5 Fake Images + Predicted Masks")


In [33]:
real_samples, fake_samples = collect_samples(test_loader, num_real=10, num_fake=10)
print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

import matplotlib.pyplot as plt
import numpy as np
import math

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display image and predicted-mask pairs in a grid that limits each row to three samples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample occupies two neighboring columns: one for the image and one for the predicted mask.
    """
    total = len(samples)
    cols = 3
    rows = math.ceil(total / cols)

    plt.figure(figsize=(cols * 6, rows * 4))

    for idx, s in enumerate(samples):
        row = idx // cols
        col = idx % cols

        # Convert the normalized tensor back to RGB for display.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to form a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Compute the first subplot column used by this sample inside the current row.
        base_col = col * 2

        img_pos  = row * cols * 2 + base_col + 1
        mask_pos = img_pos + 1

        # Show the original image and the image-level prediction.
        plt.subplot(rows, cols*2, img_pos)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}", fontsize=10)
        plt.axis("off")

        # Show the corresponding thresholded localization mask.
        plt.subplot(rows, cols*2, mask_pos)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Mask", fontsize=10)
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

show_image_and_mask(real_samples, "10 Real Images (3 per row)")
show_image_and_mask(fake_samples, "10 Fake Images (3 per row)")

Collected Real: 10  Fake: 10


### 10.2 Submission-Ready Prediction Panels

The final visualization block packages the qualitative outputs into a compact four-panel format.
Each row aligns the original image, ground-truth mask, predicted mask, and overlay so the assignment reviewer can inspect localization quality quickly.

**Assignment alignment:** This subsection supports the final qualitative presentation requirement of the assignment.


In [34]:
import matplotlib.pyplot as plt
import numpy as np

def show_submission_prediction_grid(samples, title, max_items=6):
    """
    Purpose:
        Create a submission-style four-panel view for a small set of qualitative examples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.
        max_items (int): Maximum number of examples to visualize.

    Returns:
        matplotlib.figure.Figure | None: The generated figure when samples are available.

    Notes:
        Each row shows the original image, ground-truth mask, predicted mask, and overlay for side-by-side review.
    """
    chosen = samples[:max_items]
    if not chosen:
        print(f"No samples available for: {title}")
        return None

    fig, axes = plt.subplots(len(chosen), 4, figsize=(16, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    column_titles = ["Original Image", "Ground Truth Mask", "Predicted Mask", "Overlay"]

    for row_idx, sample in enumerate(chosen):
        img = denormalize(sample["image"]).permute(1, 2, 0).numpy()
        gt_mask = (sample["mask_true"][0].numpy() > 0.5).astype(np.uint8)
        pred_mask = (sample["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Highlight predicted tampered pixels in red to make the localization decision easy to inspect.
        overlay[pred_mask == 1] = [1, 0, 0]
        blended = 0.6 * img + 0.4 * overlay

        panels = [img, gt_mask, pred_mask, blended]
        cmaps = [None, "gray", "gray", None]

        for col_idx, panel in enumerate(panels):
            ax = axes[row_idx, col_idx]
            if cmaps[col_idx] is None:
                ax.imshow(panel)
            else:
                ax.imshow(panel, cmap=cmaps[col_idx])
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(column_titles[col_idx])

        lbl = "Authentic" if sample["label"] == 0 else "Tampered"
        pred_lbl = "Authentic" if sample["pred"] == 0 else "Tampered"
        axes[row_idx, 0].set_ylabel(f"GT: {lbl}\nPred: {pred_lbl}", fontsize=10)

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()
    return fig

real_fig = show_submission_prediction_grid(real_samples, "Submission Grid: Authentic Image Predictions", max_items=4)
fake_fig = show_submission_prediction_grid(fake_samples, "Submission Grid: Tampered Image Predictions", max_items=4)

if WANDB_ACTIVE:
    if real_fig is not None:
        wandb.log({"submission_grid_authentic": wandb.Image(real_fig)})
    if fake_fig is not None:
        wandb.log({"submission_grid_tampered": wandb.Image(fake_fig)})

## 13. Inference Examples

A self-contained inference function for running the trained model on
individual images after training.


In [35]:
def predict_single_image(image_path, model, device, threshold=None):
    """Run inference on a single image and return classification + segmentation mask."""
    if threshold is None:
        threshold = CONFIG['seg_threshold']

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    transform = get_valid_transform()
    augmented = transform(image=image, mask=np.zeros(image.shape[:2], dtype='float32'))
    img_tensor = augmented['image'].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(img_tensor)

    cls_pred = torch.argmax(cls_logits, dim=1).item()
    seg_prob = torch.sigmoid(seg_logits).cpu().squeeze()
    seg_mask = (seg_prob > threshold).numpy().astype(np.uint8)

    return {
        'classification': 'tampered' if cls_pred == 1 else 'authentic',
        'confidence': torch.softmax(cls_logits, dim=1).max().item(),
        'mask_probability': seg_prob.numpy(),
        'mask_binary': seg_mask,
    }

print('Inference function defined.')
print('Usage: result = predict_single_image("path/to/image.jpg", model, device)')

Inference function defined.
Usage: result = predict_single_image("path/to/image.jpg", model, device)


## Conclusion

This notebook (vK.10.3) presents a complete, engineering-upgraded pipeline for tampered
image detection and localization.  All code runs in one notebook compatible with
Kaggle and Google Colab.

**Engineering improvements over vK.7.1:**
- Removed duplicate prior experiment block (data leakage fix)
- Centralized CONFIG dictionary for all hyperparameters
- Full reproducibility seeding across Python, NumPy, and PyTorch
- Automatic Mixed Precision (AMP) for faster training
- Three-file checkpoint system with seamless resume
- Early stopping based on tampered-only Dice coefficient
- Tampered-only metric reporting to address metric inflation
- GPU diagnostics with VRAM-based batch size auto-adjustment
- Metadata caching to skip redundant dataset scanning
- Optimized DataLoaders with persistent workers and seeded sampling

**Assignment coverage:**
- Dataset: authentic images, tampered images, ground truth masks
- Model: image-level detection + pixel-level localization
- Evaluation: Dice, IoU, F1 (all-sample and tampered-only)
- Visualization: Original, Ground Truth, Predicted Mask, Overlay panels


In [36]:
if WANDB_ACTIVE and WANDB_RUN is not None:
    WANDB_RUN.finish()
    print('W&B run finished.')
else:
    print('No active W&B run to finish.')

wandb: updating run metadata
wandb: uploading history steps 100-102, summary, console lines 443-447
wandb: 
wandb: Run history:
wandb:          epoch ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
wandb:             lr ███▇▇▆▆▆▅▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▂▂▂▃▃▄▅▆▆▇▇▇███
wandb: train/accuracy ▁▂▃▃▄▄▅▅▅▅▆▇▇▇▇████████████▇▇▇▇█▇▇▇█████
wandb:     train/dice ▁▁▁▁▁▁▃▃▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█
wandb:     train/loss █▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val/accuracy ▁▁▁▂▃▃▄▂▄▄▄▆▆▅▅▅▆▅▇▇▇▇▇▇▇▇▇▇▇▆▇█▆▇█▆▇█▄█
wandb:       val/dice █████▇▇▄▅█▄▃▄▄▄▄▅▄▄▄▄▄▄▄▅▅▇▄▆▅▅▅▄▃▅▁▅▅▄▃
wandb:         val/f1 ▇██████▆▅▁▂▃▃▂▂▃▂▂▂▃▃▃▃▁▃▄▂▄▆▅▅▄▂▄▁▂▄▄▄▂
wandb:        val/iou █████▆▇▅█▆▅▆▄▄▄▄▄▅▄▄▄▅▅▄▄▂▅▄▅▇▄▆▄▃▅▁▄▁▃▁
wandb:       val/loss █▆▅▅▅▆▅▅▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▂▂▁▂▁▂▂▁▁▂▁▁▃▂▁▁
wandb:             +4 ...
wandb: 
wandb: Run summary:
wandb:         best_epoch 96
wandb:              epoch 100
wandb:                 lr 0.0001
wandb:      test/accuracy 0.83043
wandb:          test/dice 0.42976
wandb:       test/roc_auc 0.

W&B run finished.
